# Bus Assignment in Pure Python
## Exact Methods for a Cyclic Integer Model

This notebook presents five exact approaches for the bus assignment problem in pure Python:

1. Naive enumeration
2. Backtracking with pruning
3. Constraint-driven reduced search
4. Dynamic programming
5. Branch and Bound

We solve the 6-shift model where each bus works 8 consecutive hours, so every shift covers exactly two adjacent 4-hour intervals.

Each method reports:
- one optimal assignment
- the total number of buses
- the execution time

At the end we also verify that the model has multiple optimal solutions, including the textbook solution from the notes.


In [1]:
from __future__ import annotations

from functools import lru_cache, wraps
from itertools import combinations, product
from math import ceil, floor, isclose
from time import perf_counter


In [2]:
EPSILON = 1e-9


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper


In [3]:
def dot_product(vector_a, vector_b):
    return sum(a * b for a, b in zip(vector_a, vector_b))


def is_integer_value(value: float) -> bool:
    return isclose(value, round(value), abs_tol=EPSILON)


def total_buses(values):
    return sum(values)


def solution_from_values(values, variable_names):
    return {
        **{name: int(value) for name, value in zip(variable_names, values)},
        'total_buses': int(total_buses(values)),
    }


def solve_linear_system(matrix, rhs):
    n = len(rhs)
    augmented = [row[:] + [rhs_value] for row, rhs_value in zip(matrix, rhs)]

    for col in range(n):
        pivot_row = max(range(col, n), key=lambda row: abs(augmented[row][col]))
        if abs(augmented[pivot_row][col]) < EPSILON:
            return None

        augmented[col], augmented[pivot_row] = augmented[pivot_row], augmented[col]
        pivot = augmented[col][col]

        for j in range(col, n + 1):
            augmented[col][j] /= pivot

        for row in range(n):
            if row == col:
                continue
            factor = augmented[row][col]
            for j in range(col, n + 1):
                augmented[row][j] -= factor * augmented[col][j]

    return [augmented[i][n] for i in range(n)]


def solve_lp_relaxation_by_vertices_minimize(objective, constraints, bounds):
    number_of_variables = len(objective)
    candidate_equalities = []

    for coefficients, rhs in constraints:
        candidate_equalities.append((coefficients[:], rhs))

    for i, (lower_bound, upper_bound) in enumerate(bounds):
        unit_vector = [0.0] * number_of_variables
        unit_vector[i] = 1.0
        candidate_equalities.append((unit_vector[:], lower_bound))
        candidate_equalities.append((unit_vector[:], upper_bound))

    best_point = None
    best_value = float('inf')

    for selected_equalities in combinations(candidate_equalities, number_of_variables):
        matrix = [list(eq[0]) for eq in selected_equalities]
        rhs = [eq[1] for eq in selected_equalities]

        solution = solve_linear_system(matrix, rhs)
        if solution is None:
            continue

        feasible = True
        for value, (lower_bound, upper_bound) in zip(solution, bounds):
            if value < lower_bound - EPSILON or value > upper_bound + EPSILON:
                feasible = False
                break

        if not feasible:
            continue

        for coefficients, rhs_value in constraints:
            if dot_product(coefficients, solution) < rhs_value - EPSILON:
                feasible = False
                break

        if not feasible:
            continue

        objective_value = dot_product(objective, solution)
        if objective_value < best_value - EPSILON:
            best_value = objective_value
            best_point = solution

    if best_point is None:
        return None, None

    return best_point, best_value


In [4]:
def branch_and_bound_minimize(objective, constraints, root_bounds, variable_names):
    best_integer_value = float('inf')
    best_integer_solution = None
    stack = [root_bounds]

    while stack:
        bounds = stack.pop()

        lp_solution, lp_value = solve_lp_relaxation_by_vertices_minimize(
            objective=objective,
            constraints=constraints,
            bounds=bounds,
        )

        if lp_solution is None:
            continue

        if lp_value >= best_integer_value - EPSILON:
            continue

        fractional_index = None
        for i, value in enumerate(lp_solution):
            if not is_integer_value(value):
                fractional_index = i
                break

        if fractional_index is None:
            integer_solution = [int(round(x)) for x in lp_solution]
            integer_value = dot_product(objective, integer_solution)

            if integer_value < best_integer_value - EPSILON:
                best_integer_value = integer_value
                best_integer_solution = {
                    name: value for name, value in zip(variable_names, integer_solution)
                }
                best_integer_solution['total_buses'] = int(round(integer_value))
            continue

        fractional_value = lp_solution[fractional_index]
        lower_branch_upper_bound = floor(fractional_value)
        upper_branch_lower_bound = ceil(fractional_value)

        left_bounds = [tuple(bound) for bound in bounds]
        left_lb, left_ub = left_bounds[fractional_index]
        left_bounds[fractional_index] = (
            left_lb,
            min(left_ub, float(lower_branch_upper_bound)),
        )
        if left_bounds[fractional_index][0] <= left_bounds[fractional_index][1] + EPSILON:
            stack.append(left_bounds)

        right_bounds = [tuple(bound) for bound in bounds]
        right_lb, right_ub = right_bounds[fractional_index]
        right_bounds[fractional_index] = (
            max(right_lb, float(upper_branch_lower_bound)),
            right_ub,
        )
        if right_bounds[fractional_index][0] <= right_bounds[fractional_index][1] + EPSILON:
            stack.append(right_bounds)

    return best_integer_solution


## Problem: Bus Assignment

Decision variables:
- `x1`: buses assigned from 12 a.m. to 8 a.m.
- `x2`: buses assigned from 4 a.m. to 12 p.m.
- `x3`: buses assigned from 8 a.m. to 4 p.m.
- `x4`: buses assigned from 12 p.m. to 8 p.m.
- `x5`: buses assigned from 4 p.m. to 12 a.m.
- `x6`: buses assigned from 8 p.m. to 4 a.m.

Objective:
- minimize the total number of buses assigned during the day

Coverage constraints:
- every 4-hour interval must be covered by the two shifts that overlap that interval


In [5]:
SHIFT_KEYS = [f'x{i}' for i in range(1, 7)]
SHIFT_LABELS = [
    '12 a.m. - 8 a.m.',
    '4 a.m. - 12 p.m.',
    '8 a.m. - 4 p.m.',
    '12 p.m. - 8 p.m.',
    '4 p.m. - 12 a.m.',
    '8 p.m. - 4 a.m.',
]

INTERVAL_LABELS = [
    '12 a.m. - 4 a.m.',
    '4 a.m. - 8 a.m.',
    '8 a.m. - 12 p.m.',
    '12 p.m. - 4 p.m.',
    '4 p.m. - 8 p.m.',
    '8 p.m. - 12 a.m.',
]

INTERVAL_DEMANDS = [14, 6, 12, 5, 11, 8]
SHIFT_UPPER_BOUNDS = [14, 12, 12, 11, 11, 14]

COVER_CONSTRAINTS = [
    ([1, 0, 0, 0, 0, 1], 14),
    ([1, 1, 0, 0, 0, 0], 6),
    ([0, 1, 1, 0, 0, 0], 12),
    ([0, 0, 1, 1, 0, 0], 5),
    ([0, 0, 0, 1, 1, 0], 11),
    ([0, 0, 0, 0, 1, 1], 8),
]

ROOT_BOUNDS = [(0.0, float(upper_bound)) for upper_bound in SHIFT_UPPER_BOUNDS]
OBJECTIVE = [1, 1, 1, 1, 1, 1]

TEXTBOOK_VALUES = [14, 9, 3, 3, 8, 0]
ALTERNATIVE_VALUES = [0, 6, 6, 0, 11, 14]


def is_feasible(values):
    return all(dot_product(coefficients, values) >= rhs for coefficients, rhs in COVER_CONSTRAINTS)


def check_named_solution(values):
    solution = solution_from_values(values, SHIFT_KEYS)
    return {
        'solution': solution,
        'feasible': is_feasible(values),
        'optimal_total': solution['total_buses'] == 37,
    }


In [6]:
@timed
def solve_bus_assignment_naive():
    best_total = float('inf')
    best_values = None

    for values in product(*(range(bound + 1) for bound in SHIFT_UPPER_BOUNDS)):
        if not is_feasible(values):
            continue

        total = total_buses(values)
        if total < best_total:
            best_total = total
            best_values = values

    return solution_from_values(best_values, SHIFT_KEYS)


In [7]:
@timed
def solve_bus_assignment_backtracking():
    best_total = float('inf')
    best_values = None

    chain_demands = [6, 12, 5, 11, 8]
    cycle_demand = 14

    def optimistic_completion(next_index, previous_value, first_value):
        additional_total = 0
        simulated_previous = previous_value

        for idx in range(next_index, 5):
            minimum_value = max(0, chain_demands[idx - 1] - simulated_previous)
            additional_total += minimum_value
            simulated_previous = minimum_value

        if next_index <= 5:
            last_value = max(0, chain_demands[4] - simulated_previous, cycle_demand - first_value)
            additional_total += last_value

        return additional_total

    def backtrack(index, current_values, current_total):
        nonlocal best_total, best_values

        if index == 6:
            if current_values[0] + current_values[5] < cycle_demand:
                return
            if current_total < best_total:
                best_total = current_total
                best_values = current_values[:]
            return

        if index == 0:
            start_value = 0
        else:
            start_value = max(0, chain_demands[index - 1] - current_values[index - 1])

        for value in range(start_value, SHIFT_UPPER_BOUNDS[index] + 1):
            current_values[index] = value
            new_total = current_total + value

            if index == 5 and current_values[0] + current_values[5] < cycle_demand:
                continue

            if index < 5:
                lower_bound = optimistic_completion(index + 1, value, current_values[0])
                if new_total + lower_bound >= best_total:
                    continue

            backtrack(index + 1, current_values, new_total)

        current_values[index] = 0

    backtrack(0, [0] * 6, 0)
    return solution_from_values(best_values, SHIFT_KEYS)


In [8]:
@timed
def solve_bus_assignment_reduced_search():
    best_total = float('inf')
    best_values = None

    for x3 in range(SHIFT_UPPER_BOUNDS[2] + 1):
        x2 = max(0, 12 - x3)

        for x4 in range(max(0, 5 - x3), SHIFT_UPPER_BOUNDS[3] + 1):
            x5 = max(0, 11 - x4)
            lower_x1 = max(0, 6 - x2)
            lower_x6 = max(0, 8 - x5)

            if lower_x1 + lower_x6 > 14:
                continue

            x1 = 14 - lower_x6
            x6 = lower_x6
            values = [x1, x2, x3, x4, x5, x6]

            if not is_feasible(values):
                continue

            total = total_buses(values)
            if total < best_total:
                best_total = total
                best_values = values

    return solution_from_values(best_values, SHIFT_KEYS)


In [9]:
@timed
def solve_bus_assignment_dp():
    best_total = float('inf')
    best_values = None
    chain_demands = [6, 12, 5, 11, 8]
    cycle_demand = 14

    for x1 in range(SHIFT_UPPER_BOUNDS[0] + 1):
        @lru_cache(maxsize=None)
        def dp(next_index, previous_value):
            if next_index == 5:
                x6 = max(0, chain_demands[4] - previous_value, cycle_demand - x1)
                if x6 > SHIFT_UPPER_BOUNDS[5]:
                    return float('inf'), None
                return x6, (x6,)

            minimum_value = max(0, chain_demands[next_index - 1] - previous_value)
            best_suffix_total = float('inf')
            best_suffix_values = None

            for current_value in range(minimum_value, SHIFT_UPPER_BOUNDS[next_index] + 1):
                suffix_total, suffix_values = dp(next_index + 1, current_value)
                total = current_value + suffix_total

                if total < best_suffix_total:
                    best_suffix_total = total
                    best_suffix_values = (current_value,) + suffix_values

            return best_suffix_total, best_suffix_values

        suffix_total, suffix_values = dp(1, x1)
        total = x1 + suffix_total

        if total < best_total:
            best_total = total
            best_values = [x1, *suffix_values]

    return solution_from_values(best_values, SHIFT_KEYS)


In [10]:
@timed
def solve_bus_assignment_branch_and_bound():
    return branch_and_bound_minimize(
        objective=OBJECTIVE,
        constraints=COVER_CONSTRAINTS,
        root_bounds=ROOT_BOUNDS,
        variable_names=SHIFT_KEYS,
    )


In [11]:
naive_result, naive_time = solve_bus_assignment_naive()
backtracking_result, backtracking_time = solve_bus_assignment_backtracking()
reduced_result, reduced_time = solve_bus_assignment_reduced_search()
dp_result, dp_time = solve_bus_assignment_dp()
bb_result, bb_time = solve_bus_assignment_branch_and_bound()

print('=== BUS ASSIGNMENT RESULTS ===')
print(f'Naive               -> {naive_result}, time = {naive_time:.8f} seconds')
print(f'Backtracking        -> {backtracking_result}, time = {backtracking_time:.8f} seconds')
print(f'Reduced Search      -> {reduced_result}, time = {reduced_time:.8f} seconds')
print(f'Dynamic Programming -> {dp_result}, time = {dp_time:.8f} seconds')
print(f'Branch and Bound    -> {bb_result}, time = {bb_time:.8f} seconds')


=== BUS ASSIGNMENT RESULTS ===
Naive               -> {'x1': 0, 'x2': 6, 'x3': 6, 'x4': 0, 'x5': 11, 'x6': 14, 'total_buses': 37}, time = 6.93354271 seconds
Backtracking        -> {'x1': 0, 'x2': 6, 'x3': 6, 'x4': 0, 'x5': 11, 'x6': 14, 'total_buses': 37}, time = 0.00003883 seconds
Reduced Search      -> {'x1': 12, 'x2': 12, 'x3': 0, 'x4': 5, 'x5': 6, 'x6': 2, 'total_buses': 37}, time = 0.00041579 seconds
Dynamic Programming -> {'x1': 0, 'x2': 6, 'x3': 6, 'x4': 0, 'x5': 11, 'x6': 14, 'total_buses': 37}, time = 0.00055925 seconds
Branch and Bound    -> {'x1': 6, 'x2': 0, 'x3': 12, 'x4': 11, 'x5': 0, 'x6': 8, 'total_buses': 37}, time = 0.18680975 seconds


## Multiple Optimal Solutions

Because this is a covering model with symmetric tradeoffs between adjacent shifts, different assignments can achieve the same minimum total.

The textbook solution reported in the notes is one optimum, but it is not unique.


In [12]:
def enumerate_all_optimal_solutions():
    optimal_total = 37
    solutions = []

    for x3 in range(SHIFT_UPPER_BOUNDS[2] + 1):
        x2 = max(0, 12 - x3)

        for x4 in range(max(0, 5 - x3), SHIFT_UPPER_BOUNDS[3] + 1):
            x5 = max(0, 11 - x4)
            lower_x1 = max(0, 6 - x2)
            lower_x6 = max(0, 8 - x5)

            for x1 in range(lower_x1, 15 - lower_x6):
                x6 = optimal_total - (x1 + x2 + x3 + x4 + x5)
                if x6 < lower_x6 or x6 > SHIFT_UPPER_BOUNDS[5]:
                    continue

                values = [x1, x2, x3, x4, x5, x6]
                if total_buses(values) != optimal_total or not is_feasible(values):
                    continue

                solutions.append(solution_from_values(values, SHIFT_KEYS))

    return solutions


optimal_solutions = enumerate_all_optimal_solutions()
textbook_check = check_named_solution(TEXTBOOK_VALUES)
alternative_check = check_named_solution(ALTERNATIVE_VALUES)

print('=== MULTIPLE OPTIMA ===')
print(f"Optimal total buses           -> {optimal_solutions[0]['total_buses']}")
print(f'Number of optimal solutions  -> {len(optimal_solutions)}')
print(f'Textbook solution check      -> {textbook_check}')
print(f'Alternative solution check   -> {alternative_check}')
print()
print('First 5 optimal solutions:')
for solution in optimal_solutions[:5]:
    print(solution)


=== MULTIPLE OPTIMA ===
Optimal total buses           -> 37
Number of optimal solutions  -> 1396
Textbook solution check      -> {'solution': {'x1': 14, 'x2': 9, 'x3': 3, 'x4': 3, 'x5': 8, 'x6': 0, 'total_buses': 37}, 'feasible': True, 'optimal_total': True}
Alternative solution check   -> {'solution': {'x1': 0, 'x2': 6, 'x3': 6, 'x4': 0, 'x5': 11, 'x6': 14, 'total_buses': 37}, 'feasible': True, 'optimal_total': True}

First 5 optimal solutions:
{'x1': 0, 'x2': 12, 'x3': 0, 'x4': 5, 'x5': 6, 'x6': 14, 'total_buses': 37}
{'x1': 1, 'x2': 12, 'x3': 0, 'x4': 5, 'x5': 6, 'x6': 13, 'total_buses': 37}
{'x1': 2, 'x2': 12, 'x3': 0, 'x4': 5, 'x5': 6, 'x6': 12, 'total_buses': 37}
{'x1': 3, 'x2': 12, 'x3': 0, 'x4': 5, 'x5': 6, 'x6': 11, 'total_buses': 37}
{'x1': 4, 'x2': 12, 'x3': 0, 'x4': 5, 'x5': 6, 'x6': 10, 'total_buses': 37}
